# Занятие 5, пара 2. Сопоставление, гомография и «скан из фото»


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## От «похоже» к «находится на своём месте»

Телефон умеет превращать косой снимок листа в прямоугольник. Здесь не возникает новая
информация о тексте: меняется геометрия уже имеющихся пикселей. Разберём, что для этого нужно знать.

Сначала увидим, почему ближайший дескриптор может ошибиться. Затем проверим совпадения общей
геометрией и выровняем учебный лист по четырём соответствующим углам.
**Сопоставление** предлагает пары точек, **геометрическая модель** проверяет, могут ли они
описывать одно преобразование. В конце углы будут заданы явно — мы не выдаём ручной выбор за
полностью автоматический сканер.

Нужны: дескриптор и расстояние из 05.1, координаты(x,y), работа с матрицами на уровне умножения.
Основной путь — NumPy/Matplotlib. Все примеры создаются заново; первая тетрадь в ядре не нужна.
Результат — выровненный учебный лист, ошибка по контрольным точкам и объяснение ограничения плоской модели.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check, summary

PROJECT_DIR


### 1. Ближайший кандидат — ещё не надёжный

Для прозрачности возьмём дескрипторы длины 3. В реальном изображении это более длинные списки,
но правило расстояния то же. `np.argsort` возвращает **индексы** элементов в порядке возрастания:
с его помощью найдём ближайшего и второго по близости кандидата.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

query = np.array([0.2,0.8,0.1])
candidates = np.array([[0.21,0.79,0.1], [0.22,0.78,0.1], [0.8,0.1,0.4]])
distances = []
for candidate in candidates:
    distances.append(np.linalg.norm(query - candidate))
order = np.argsort(distances)
print('Расстояния:', distances)
print('Индексы двух ближайших:', order[:2])


Первые два кандидата похожи. Если лучший лишь чуть лучше второго, выбор ненадёжен.
**Тест отношения расстояний** принимает пару, когда `d1 < ratio * d2`, где d1≤d2.
Возьмём ratio=0.75 как учебный порог, а не универсальную константу. При меньшем пороге отбор строже.
В отличие от простой границы d1, сравниваем уверенность выбора среди кандидатов.
Запись без деления безопасна при d2=0: два идеально одинаковых кандидата будут отвергнуты.


In [ ]:
d1 = distances[order[0]]
d2 = distances[order[1]]
ratio = 0.75
accepted = d1 < ratio * d2
print('Лучшая пара проходит:', accepted)
print('Два одинаковых кандидата:', 0 < ratio * 0)


Этот тест может отбросить полезную пару или сохранить ложную. Он не проверяет геометрию,
не является вероятностью правильного ответа и требует хотя бы двух кандидатов.
[Сопоставление и тест отношения в OpenCV](https://docs.opencv.org/4.x/dc/dc3/tutorial_py_matcher.html).


### ✏️ Ваш ход — сделать отбор осмысленным

Есть 4 пары расстояний `pairs = np.array([[1.,4.],[3.,3.2],[0.,0.],[2.,3.]])`.
Первый столбец — лучший кандидат, второй — следующий. Получите маску `my_keep` при ratio0.75.
Посчитайте `my_fraction` — долю принятых. Объясните решение для строки[0,0].


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('05.2.1', my_keep, my_fraction)


**Мой вывод:** …


### 2. Проверим общую геометрию на простой модели

Представим, что второй кадр просто сдвинули на(dx,dy). Тогда у правильных пар разность
координат должна быть примерно одной и той же. В учебном наборе 5 правильных пар и 2 ложных.
Мы знаем это потому, что создаём опыт, а не потому, что алгоритм заранее знает ответ.
Попробуем сначала средний сдвиг всех пар — и увидим, как ошибки тянут его в сторону.


In [ ]:
source = np.array([[10.,10.],[30.,10.],[10.,30.],[30.,30.],[20.,20.],[5.,40.],[40.,5.]])
target = source + np.array([12.,-4.])
target[5] = [70.,10.]
target[6] = [5.,70.]
offsets = target - source
mean_shift = offsets.mean(axis=0)
print('Сдвиги пар:')
print(offsets)
print('Средний сдвиг:', mean_shift)


**RANSAC** пробует гипотезу по маленькой случайной части данных, считает согласные пары
и выбирает вариант с большой поддержкой. Согласные называют inliers, несогласные — outliers.
Для **переноса** достаточно одной пары. Для гомографии понадобится минимум 4 при невырожденном расположении.
Ниже маленькая учебная версия только переноса, не универсальный RANSAC для гомографии.

`default_rng(7)` фиксирует генератор, чтобы опыт повторялся; `integers` выбирает индекс пары.
`norm(..., axis=1)` считает расстояние отдельно для каждой строки(x,y).
Порог 1 пиксель задаёт допустимую ошибку положения — это уже не порог расстояния дескрипторов.


In [ ]:
rng = np.random.default_rng(7)
best_count = -1
best_mask = None
for attempt in range(30):
    index = rng.integers(len(source))
    hypothesis = offsets[index]
    predicted = source + hypothesis
    errors = np.linalg.norm(predicted - target, axis=1)
    inliers = errors < 1.0
    count = np.count_nonzero(inliers)
    if count > best_count:
        best_count = count
        best_mask = inliers
robust_shift = offsets[best_mask].mean(axis=0)
print('Согласных пар:', best_count, 'из', len(source))
print('Сдвиг по согласным:', robust_shift)


Нашлись 5 пар и сдвиг(12,−4). RANSAC не «исправил» ложные соответствия — он исключил их
из оценки этой модели. Если правильных пар мало, порог неудачен или сама модель неверна,
метод тоже может ошибиться. Общий сдвиг не объяснит перспективное изменение листа.

**Обсудите 2 минуты:** что произойдёт, если вместо переноса повернуть карточку? Сохранится ли
одна разность координат у всех её точек?


### 3. Для плоскости нужна гомография

На косом снимке прямоугольный лист выглядит четырёхугольником. Его линии остаются прямыми,
но параллельные стороны на фото могут сходиться. **Гомография** — преобразование координат
между двумя изображениями одной плоскости. Для практики считаем лист плоским, а искажение линзы
небольшим. При сильной дисторсии сначала нужна коррекция камеры из блока 02.

Гомография не распрямляет согнутую книгу и не описывает произвольную сцену с разной глубиной
одной матрицей. Она также не восстанавливает закрытые или нечёткие буквы.


К точке(x,y) допишем 1. Умножим этот столбец на матрицу 3×3:

$$
\begin{bmatrix}u'\\v'\\w'\end{bmatrix}
=H\begin{bmatrix}x\\y\\1\end{bmatrix},\qquad
u=u'/w',\quad v=v'/w'.
$$

Тройка — **однородные координаты**. Последнее деление обязательно: именно знаменатель,
зависящий от места, позволяет описать перспективу. Матрицы H и 2H задают одно преобразование.
Знак `@` в NumPy — матричное умножение, не поэлементное `*`. Начнём с обычного переноса.


In [ ]:
H_shift = np.array([[1.,0.,12.], [0.,1.,-4.], [0.,0.,1.]])
point = np.array([10.,20.,1.])
mapped = H_shift @ point
mapped_xy = mapped[:2] / mapped[2]
print('После переноса:', mapped_xy)


Для перспективы изменим нижнюю строку. Сейчас мы **задаём** модель, ещё не оцениваем её
по данным. Функция ниже повторяет разобранное действие для списка точек.
`column_stack` добавляет колонку единиц, `.T` разворачивает строки в столбцы для умножения.
`mapped[:,2:3]` оставляет знаменатель столбцом, чтобы делить обе координаты каждой строки.
Точки с почти нулевым знаменателем лежат у линии, которую модель отправляет очень далеко;
функция явно остановится вместо выдачи бессмысленных координат.


In [ ]:
def project_points(points, H):
    points = np.asarray(points, dtype=float)
    homogeneous = np.column_stack([points, np.ones(len(points))])
    mapped = (H @ homogeneous.T).T
    if np.any(np.abs(mapped[:,2]) < 1e-10):
        raise ValueError('Точка слишком близко к нулевому знаменателю')
    return mapped[:,:2] / mapped[:,2:3]

H_perspective = np.array([[1.,0.15,25.], [0.08,1.1,20.], [0.002,0.001,1.]])
print(project_points([[0,0], [100,0], [100,80]], H_perspective))


### ✏️ Ваш ход — не забыть последнюю строку

Для точки(10,20) и матрицы
`[[2.,0.,0.],[0.,2.,0.],[0.1,0.,1.]]` вычислите однородный результат в `my_homogeneous`,
а обычные координаты в `my_xy`. Сначала вручную: чему равен знаменатель?
Можно проверить через project_points, но покажите отдельно умножение и деление.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('05.2.2', my_homogeneous, my_xy)


**Мой вывод:** …


### 4. Сделаем контролируемую «фотографию»

Загрузим учебный лист. `warp_nearest` из поддержки — готовая утилита пересчёта пикселей:
входы — серое изображение, H из исходника в результат и размер(высота,ширина);
выходы — новое изображение и маска мест, покрытых исходником. Она не ищет точки и не находит H.
Для каждого выходного пикселя берётся ближайший пиксель исходника; от этого возможны ступеньки.
Сначала используем её как инструмент, ниже разберём направление этого пересчёта.


In [ ]:
from course_support.geometry import warp_nearest

document = np.load(DATA_DIR / 'features' / 'document.npy', allow_pickle=False)
photo, photo_coverage = warp_nearest(document, H_perspective, (160,200), fill=0.15)
doc_height, doc_width = document.shape
document_corners = np.array([[0.,0.],[doc_width-1.,0.],[doc_width-1.,doc_height-1.],[0.,doc_height-1.]])
photo_corners = project_points(document_corners, H_perspective)
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].imshow(document, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('Исходный синтетический лист')
axes[1].imshow(photo, cmap='gray', vmin=0, vmax=1)
axes[1].scatter(photo_corners[:,0], photo_corners[:,1], c='red')
axes[1].set_title('Созданная нами перспектива, не реальное фото')
plt.show()


Порядок углов: верхний левый, верхний правый, нижний правый, нижний левый **на листе**.
После преобразования сохраняем соответствие индексов. Если перепутать два угла, матрица будет
решать уже другую задачу. Здесь `photo_corners` известны из модели; при ручном сканировании
их отмечает человек, при автоматическом — алгоритм с отдельными ошибками поиска.

Почему 4 пары? У H9 чисел, но общий множитель не важен: остаётся 8 степеней свободы.
Одна пара(x,y)→(u,v) даёт два уравнения. Для 4 пар получаем 8 уравнений.
Это работает при невырожденном расположении: например, четыре точки на одной прямой не подойдут.


### 5. Найдём H из соответствий

В нашем ограниченном учебном решении фиксируем нижний правый элемент H равным 1.
После переноса знаменателя каждое соответствие даёт две строки системы:

$$
xh_{11}+yh_{12}+h_{13}-uxh_{31}-uyh_{32}=u,
$$
$$
xh_{21}+yh_{22}+h_{23}-vxh_{31}-vyh_{32}=v.
$$

Сначала соберём две строки **для одной пары**: координаты фото — исходные, координаты ровного
листа — целевые. Мы ищем обратное направление по сравнению с созданием фотографии.


In [ ]:
x, y = photo_corners[0]
u, v = document_corners[0]
row_u = [x,y,1,0,0,0,-u*x,-u*y]
row_v = [0,0,0,x,y,1,-v*x,-v*y]
print('Коэффициенты первого уравнения:', row_u, 'ответ:', u)
print('Коэффициенты второго уравнения:', row_v, 'ответ:', v)


Теперь повторяем эти две строки для четырёх пар. `np.linalg.solve(A,b)` решает квадратную
систему `A @ h = b`. При вырожденной системе будет ошибка; она не означает, что нужно
добавить случайное число в матрицу. Проверяют точки и модель.
`np.append(h,1).reshape(3,3)` дописывает фиксированную 1 и возвращает форму матрицы.
В промышленном оценивании используют более устойчивые методы, нормализацию и много пар;
наша функция — понятный расчёт по 4 точным соответствиям, не замена findHomography на любых данных.


In [ ]:
def homography_four(source_points, target_points):
    rows = []
    right_side = []
    for (x,y), (u,v) in zip(source_points, target_points):
        rows.append([x,y,1,0,0,0,-u*x,-u*y])
        rows.append([0,0,0,x,y,1,-v*x,-v*y])
        right_side.extend([u,v])
    coefficients = np.asarray(rows, dtype=float)
    answers = np.asarray(right_side, dtype=float)
    h = np.linalg.solve(coefficients, answers)
    return np.append(h,1).reshape(3,3)

H_scan = homography_four(photo_corners, document_corners)
print('Фото → лист:')
print(H_scan)


Проверим, куда модель отправила заданные углы. Расстояние между предсказанной и нужной
точкой — **ошибка репроекции**, здесь в пикселях ровного листа.
Почти нулевая ошибка на этих же 4 точках ожидаема: на них мы подогнали 8 параметров.
Она ещё не доказывает хорошую работу на других точках или правильность ручной разметки.


In [ ]:
predicted_corners = project_points(photo_corners, H_scan)
corner_errors = np.linalg.norm(predicted_corners - document_corners, axis=1)
print('Ошибки четырёх опорных углов:', corner_errors)


### 6. Переносим пиксели, а не только углы

Если двигать каждый исходный пиксель вперёд, некоторые выходные клетки останутся дырками.
Поэтому для **каждой клетки результата** ищут место в исходнике — это обратное отображение.
Для матрицы фото→лист нужна обратная матрица, чтобы ответить: «откуда взять яркость для этой точки листа?»
`np.linalg.inv` вычисляет обратную матрицу. Сначала проверим одну точку центра.


In [ ]:
H_back_to_photo = np.linalg.inv(H_scan)
center_on_sheet = np.array([[doc_width/2,doc_height/2]])
sampling_position = project_points(center_on_sheet, H_back_to_photo)
print('Центр листа берёт яркость около точки фото:', sampling_position)


`warp_nearest` делает такой поиск для всех клеток результата и округляет до ближайшего пикселя.
Ей передаём **прямую H фото→лист** — обращение она выполняет внутри. Размер результата
`document.shape` задаёт выбранную сетку, а не «восстанавливает истинное разрешение камеры».


In [ ]:
scan, scan_coverage = warp_nearest(photo, H_scan, document.shape, fill=0.15)
fig, axes = plt.subplots(1,3,figsize=(12,4))
for axis, picture, title in zip(axes, [document,photo,scan], ['Лист','Перспектива','Выравнивание']):
    axis.imshow(picture, cmap='gray', vmin=0, vmax=1)
    axis.set_title(title)
plt.show()


Лист снова прямоугольный. Но после двух пересчётов не обязаны точно восстановиться все
исходные пиксели: округление при уменьшении теряет информацию. Билинейная интерполяция даёт
более гладкий вид, но тоже не возвращает утраченную детализацию. Не принимаем красивую картинку
за достаточную проверку геометрии.


### ✏️ Ваш ход — проверить середины сторон

Возьмите 4 контрольные точки листа:
`[[70.,0.],[139.,50.],[70.,99.],[0.,50.]]`. Они не использовались при решении системы.
Получите их положение на учебном фото через `H_perspective`, затем верните через `H_scan`.
Вычислите 4 ошибки в `my_control_errors` как norm разностей по axis=1.
Почему эта проверка содержательнее повторной проверки тех же углов?
Это контроль синтетического опыта, не измерение точности на реальном документе.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('05.2.3', my_control_errors)


**Мой вывод:** …


### 7. Один угол отметили неточно — что сломается?

Основной опыт независим от «Ваш ход»: создаём свои контрольные точки заново.
Сдвинем только второй отмеченный угол на(8,−5)пикселей. Он не стал правильным оттого,
что система уравнений нашла решение. Сравним ошибки **на точках с известной правильной геометрией**.


In [ ]:
wrong_corners = photo_corners.copy()
wrong_corners[1] += [8,-5]
H_wrong = homography_four(wrong_corners, document_corners)
controls = np.array([[70.,0.],[139.,50.],[70.,99.],[0.,50.],[70.,50.]])
photo_controls = project_points(controls, H_perspective)
good_recovery = project_points(photo_controls, H_scan)
bad_recovery = project_points(photo_controls, H_wrong)
print('Правильная разметка:', np.linalg.norm(good_recovery-controls, axis=1))
print('Один ошибочный угол:', np.linalg.norm(bad_recovery-controls, axis=1))
wrong_scan, _ = warp_nearest(photo, H_wrong, document.shape, fill=0.15)
fig, axes = plt.subplots(1,2,figsize=(9,4))
axes[0].imshow(scan, cmap='gray', vmin=0, vmax=1); axes[0].set_title('Верные углы')
axes[1].imshow(wrong_scan, cmap='gray', vmin=0, vmax=1); axes[1].set_title('Один угол смещён')
plt.show()


**Исследование на 8 минут:** предскажите, где ошибка будет заметнее. Сравните числа и изображение.
Затем в своей копии уменьшите сдвиг угла вдвое и посмотрите, уменьшились ли ошибки.
Фраза «выглядит нормально» теперь дополняется измерением.

Для реального листа контрольные точки с известной истиной могут отсутствовать. Тогда проверяют
разумность формы, прямизну строк и устойчивость к изменению разметки; точность не объявляют
по нулевой ошибке на тех же 4 углах. Автоматические совпадения дают много пар, а RANSAC помогает
отбросить несогласные; он не устраняет неплоскость документа.


### Сохраним результат и объяснение

`plt.imsave` записывает растровую картинку. Создаём свою папку `reports`, исходники не меняем.
В notebook рядом запишите: как получили углы; направление H; максимальную ошибку контрольного
опыта; одно ограничение. Здесь достаточно выданного набора — фотографировать личные документы не нужно.


In [ ]:
report_dir = WORKSPACE_DIR / 'reports'
report_dir.mkdir(exist_ok=True)
output_path = report_dir / '05_document_scan.png'
plt.imsave(output_path, scan, cmap='gray', vmin=0, vmax=1)
print('Сохранено:', output_path)


### Дополнительное сравнение с OpenCV

Если OpenCV установлен: `getPerspectiveTransform` решает преобразование по 4 парам;
`perspectiveTransform` переносит точки; `warpPerspective` пересчитывает изображение.
В нём размер задаётся как **(ширина,высота)**, в нашей утилите — (высота,ширина).
Сравниваем координаты контрольных точек, а не требуем побитового совпадения картинок разных интерполяций.
Для большого набора шумных соответствий используют `findHomography(..., RANSAC, порог)`;
порог имеет единицы пикселей целевого изображения, а не сходства дескрипторов.


In [ ]:
try:
    import cv2
except (ImportError, OSError):
    cv2 = None

if cv2 is None:
    print('OpenCV недоступен; собственное выравнивание на NumPy выполнено выше.')
else:
    H_cv = cv2.getPerspectiveTransform(photo_corners.astype(np.float32), document_corners.astype(np.float32))
    cv_controls = project_points(photo_controls, H_cv)
    print('Расхождение NumPy/OpenCV на точках:', np.linalg.norm(cv_controls-good_recovery, axis=1))
    scan_cv = cv2.warpPerspective(photo, H_cv, (doc_width,doc_height), flags=cv2.INTER_NEAREST)
    plt.imshow(scan_cv, cmap='gray', vmin=0, vmax=1)
    plt.title('Дополнительный собственный запуск OpenCV')
    plt.show()


### Что получилось и что пока осталось за рамками

Мы предложили совпадения по расстоянию, разобрали геометрический отбор на модели переноса
и сами выровняли плоский лист по 4 парам. Это **части** будущей системы; полноценного автоматического
поиска документа в любом кадре здесь пока нет. Дескрипторы не обязаны выдерживать сильную перспективу.

**Запас на 10–15 минут:** поменяйте порядок двух целевых углов, предскажите результат,
сравните с правильным. Затем верните порядок. Другой опыт — отметьте точки почти на одной прямой
и обсудите, почему модель становится ненадёжной; не пытайтесь «лечить» её случайным шумом.

Почитать: [официальный пример сопоставления и гомографии](https://docs.opencv.org/4.x/d1/de0/tutorial_py_feature_homography.html).
В нём найдите три уже понятные части: кандидаты, отбор, преобразование. Полную панораму
из десятка кадров или декомпозицию H делать к следующей паре не требуется.
